In [2]:
# ...existing code...
import networkx as nx
import plotly.graph_objects as go

def compute_pair_stats(df, df_binary, comp_cols, price_col=None, thickness_col=None):
    import itertools
    pair_rows = []
    cols = df_binary.columns.tolist()
    for a,b in itertools.combinations(cols,2):
        mask = (df_binary[a]==1) & (df_binary[b]==1)
        cnt = int(mask.sum())
        if cnt==0:
            pair_rows.append({"pair": f"{a} + {b}", "count": 0, "mean_price": np.nan, "median_thickness": np.nan})
            continue
        mean_price = np.nan
        median_th = np.nan
        if price_col and price_col in df.columns:
            mean_price = float(pd.to_numeric(df.loc[mask, price_col], errors='coerce').dropna().mean() or np.nan)
        if thickness_col and thickness_col in df.columns:
            median_th = float(pd.to_numeric(df.loc[mask, thickness_col], errors='coerce').dropna().median() or np.nan)
        pair_rows.append({"pair": f"{a} + {b}", "count": cnt, "mean_price": mean_price, "median_thickness": median_th})
    pairs_df = pd.DataFrame(pair_rows).sort_values("count", ascending=False).reset_index(drop=True)
    return pairs_df

# Panel nuevo: pares frecuentes + stats
if not df_binary.empty:
    st.markdown("### Material pair frequency & stats")
    pairs_df = compute_pair_stats(df, df_binary, comp_cols, price_col=price_col, thickness_col=thickness_col)

    cols = st.columns([2,1])
    with cols[0]:
        st.write("Top pairs (by co-occurrence)")
        st.dataframe(pairs_df.head(20).reset_index(drop=True))

        # barras top
        fig_bar = go.Figure()
        topn = pairs_df.head(15)
        fig_bar.add_trace(go.Bar(x=topn["pair"], y=topn["count"], name="count", marker_color="teal"))
        fig_bar.update_layout(title="Top material pairs (count)", xaxis_tickangle=-45, height=420, margin=dict(t=40))
        st.plotly_chart(fig_bar, use_container_width=True)

    with cols[1]:
        st.write("Least frequent pairs (including zero)")
        st.dataframe(pairs_df.tail(20).sort_values("count").reset_index(drop=True))
        bottomn = pairs_df.tail(15).sort_values("count")
        fig_bar2 = go.Figure()
        fig_bar2.add_trace(go.Bar(x=bottomn["pair"], y=bottomn["count"], name="count", marker_color="lightgray"))
        fig_bar2.update_layout(title="Least frequent material pairs", xaxis_tickangle=-45, height=420, margin=dict(t=40))
        st.plotly_chart(fig_bar2, use_container_width=True)

    st.markdown("#### Pair price / thickness comparison")
    sel_pair = st.selectbox("Select a pair to inspect", options=pairs_df["pair"].tolist(), index=0)
    if sel_pair:
        a,b = [s.strip() for s in sel_pair.split("+")]
        mask = (df_binary[a]==1) & (df_binary[b]==1)
        sample = df.loc[mask].copy()
        st.write(f"Products with {sel_pair}: {len(sample)}")
        if not sample.empty:
            cols_show = [c for c in [name_col, price_col, thickness_col, color_col] + comp_cols if c in sample.columns]
            st.dataframe(sample[cols_show].head(50))

            # scatter price vs thickness for this pair
            if price_col and thickness_col and price_col in sample.columns and thickness_col in sample.columns:
                s = sample.copy()
                s[price_col] = pd.to_numeric(s[price_col], errors='coerce')
                s[thickness_col] = pd.to_numeric(s[thickness_col], errors='coerce')
                fig = px.scatter(s, x=thickness_col, y=price_col, hover_data=[name_col, color_col] if name_col and color_col else [name_col],
                                 title=f"Price vs Thickness for {sel_pair}", color_discrete_sequence=["purple"])
                st.plotly_chart(fig, use_container_width=True)
            else:
                st.info("Price or thickness column missing for scatter.")

    st.markdown("#### Material co-occurrence network")
    # Construir grafo donde nodos = materiales y aristas peso=count
    G = nx.Graph()
    for idx,row in pairs_df.iterrows():
        a,b = [s.strip() for s in row["pair"].split("+")]
        cnt = int(row["count"])
        if cnt>0:
            G.add_node(a, size=int(df_binary[a].sum()))
            G.add_node(b, size=int(df_binary[b].sum()))
            G.add_edge(a, b, weight=cnt)

    if len(G.nodes)>0:
        pos = nx.spring_layout(G, seed=42, k=0.6)
        edge_x = []
        edge_y = []
        weights = []
        for u,v,edata in G.edges(data=True):
            x0,y0 = pos[u]
            x1,y1 = pos[v]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            weights.append(edata.get("weight",1))

        node_x = []
        node_y = []
        node_text = []
        node_size = []
        for n,data in G.nodes(data=True):
            x,y = pos[n]
            node_x.append(x); node_y.append(y)
            node_text.append(f"{n} ({int(data.get('size',0))})")
            node_size.append(max(8, int(data.get('size',0)) * 3))

        edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=1, color='#888'), hoverinfo='none', mode='lines')
        node_trace = go.Scatter(x=node_x, y=node_y, mode='markers+text', hoverinfo='text',
                                textposition="top center",
                                marker=dict(showscale=False, color="skyblue", size=node_size, line_width=1),
                                text=[n for n in G.nodes()])
        node_trace.text = node_text

        fig_net = go.Figure(data=[edge_trace, node_trace],
                            layout=go.Layout(title="Material co-occurrence network (edge weight = count)",
                                             showlegend=False, hovermode='closest',
                                             margin=dict(b=20,l=5,r=5,t=40)))
        st.plotly_chart(fig_net, use_container_width=True)
    else:
        st.info("Not enough co-occurrence data to build network.")
# ...existing code...

ModuleNotFoundError: No module named 'networkx'